In [ ]:
from functools import partial
import matplotlib.pyplot as plt
import cupy as cp
from superfv import HydroSolver, ics, TimeIntegrator, BC
from teyssier.sedov import sedovana

In [ ]:
N = 256
sim = HydroSolver(
    ic=partial(ics.sedov, gamma=1.4, h=1 / N),
    nx=N,
    ny=N,
    nz=N,
    bcx=(BC.REFLECTIVE, BC.FREE),
    bcy=(BC.REFLECTIVE, BC.FREE),
    bcz=(BC.REFLECTIVE, BC.FREE),
    p=1,
    use_MUSCL=True,
    cupy=True,
)

In [ ]:
sim.run(0.8, time_integrator=TimeIntegrator.MUSCL_HANCOCK)

In [ ]:
n_steps = sim.step_history[-1].step
n_xyz = sim.params.mesh.nx * sim.params.mesh.ny * sim.params.mesh.nz
t_take_step = sim.step_history.get_total_time("take_step")
update_rate = n_steps * n_xyz / t_take_step

t_compute_dt = sim.step_history.get_total_time("compute_dt")
t_update_unew = sim.step_history.get_total_time("update_unew")
t_update_W = sim.step_history.get_total_time("update_W")
t_update_fluxes = sim.step_history.get_total_time("update_fluxes")
t_riemann_solver = sim.step_history.get_total_time("riemann_solver")
t_compute_dudt = sim.step_history.get_total_time("compute_dudt")

print(f"  update_rate = {update_rate:8.2e} cell/s\n")
print(f"    compute_dt: {t_compute_dt/t_take_step * 100:8.2f} %")
print(f"   update_unew: {t_update_unew/t_take_step * 100:8.2f} %")
print(f"        update_W: {t_update_W/t_take_step * 100:8.2f} %")
print(f"   update_fluxes: {t_update_fluxes/t_take_step * 100:8.2f} %")
print(f"    riemann_solver: {t_riemann_solver/t_take_step * 100:8.2f} %")
print(f"    compute_dudt: {t_compute_dudt/t_take_step * 100:8.2f} %")

In [ ]:
# analytical solution
dim = 3
E0 = 1
rho0 = 1
T = sim.step_history[-1].t_sim

r, d, u, P = sedovana(1.4, dim)

r *= (E0 / rho0) ** (1.0 / (dim + 2)) * T ** (2 / (dim + 2))
d *= rho0
u *= (E0 / rho0) ** (1.0 / (dim + 2)) * T ** (-dim / (dim + 2))
P *= (E0 / rho0) ** (2.0 / (dim + 2)) * T ** (-2 * dim / (dim + 2)) * rho0

In [ ]:
def radial_binned_avg(q, X, Y, Z, center, r_edges):
    R = cp.sqrt((X - center[0]) ** 2 + (Y - center[1]) ** 2 + (Z - center[2]) ** 2)
    k = cp.digitize(R.ravel(), r_edges)
    ok = (0 < k) & (k < len(r_edges))

    sums = cp.bincount(k[ok] - 1, weights=q.ravel()[ok], minlength=len(r_edges) - 1)
    counts = cp.bincount(k[ok] - 1, minlength=len(r_edges) - 1)

    return sums / counts


r_edges = cp.linspace(0, 1, 64)

X = sim.mesh.array_manager["core_X"]
Y = sim.mesh.array_manager["core_Y"]
Z = sim.mesh.array_manager["core_Z"]
rho = cp.asarray(sim.snapshot_history[-1].u[sim.idx("rho")])

rho_binned = cp.asnumpy(radial_binned_avg(rho, X, Y, Z, (0, 0, 0), r_edges))
r_centers = cp.asnumpy(0.5 * (r_edges[:-1] + r_edges[1:]))

In [ ]:
plt.xlim(sim.mesh.x_centers[0], sim.mesh.x_centers[-1])
plt.plot(r, d, "k")
plt.plot(r_centers, rho_binned, linestyle="--", marker=".", mfc="none")